# Understanding Your Data

Before cleaning or transforming any dataset, you need to **understand its structure**: what uniquely identifies a row, what relationships exist between tables, and where duplicates or integrity issues hide.

In **Notebook 04** we reconciled schema drift and created `schools_reconciled` and `pupils_reconciled` in silver — tables with clean, unified column names and explicit `term` and `year` fields. This notebook uses those tables to work through four diagnostic steps:

1. **Remove exact duplicates** — a mechanical first step that clears the noise
2. **Check cardinality** — verify the grain of your data
3. **Understand keys** — primary, natural, surrogate, and foreign keys
4. **Validate referential integrity** — ensure foreign keys point to real records

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

## Step 1 — Removing exact duplicates

When you try to check the grain of a table (e.g. "is `school_urn` unique?"), **exact duplicate rows** contaminate the result. Two rows that are carbon copies of each other make the key look non-unique even if the grain is correct.

Exact duplicates — rows where **every column is identical** — can be removed mechanically with no domain knowledge. This is a safe, zero-judgement first step that clears the way for meaningful cardinality checks.

We'll use `CONCAT_WS` to hash all columns into a single value, then `ROW_NUMBER` to keep only the first occurrence. Once exact duplicates are removed, we assign a `row_id` surrogate key to each surviving row for audit tracking.

In [0]:
USE CATALOG catalog_40_copper_analyst_training;

In [0]:
-- Detect exact duplicates: rows where every column is identical
-- CONCAT_WS joins all column values with a separator — identical rows produce identical strings
SELECT
  CONCAT_WS('|', *) AS row_hash
  ,COUNT(*) AS occurrences
FROM silver.schools_reconciled
GROUP BY row_hash
HAVING COUNT(*) > 1;

### How does this work?

`CONCAT_WS('|', *)` concatenates **every column** in the row into a single string, using `|` as a separator. Rows that are exact duplicates produce identical strings.

By grouping on this concatenated value and filtering to `COUNT(*) > 1`, we find all rows that appear more than once. The next step uses `ROW_NUMBER` to keep only the first occurrence of each.

> **Note:** `CONCAT_WS` handles `NULL` values safely (unlike plain `CONCAT`, which returns `NULL` if any argument is `NULL`).

In [0]:
-- Remove exact duplicates: keep only the first occurrence of each identical row
-- Assign row_id as a surrogate key after dedup so every surviving row has a stable identifier
CREATE OR REPLACE TEMP VIEW schools_no_exact_dupes AS
WITH cte_hashed AS (
  SELECT CONCAT_WS('|', *) AS row_hash, *
  FROM silver.schools_reconciled
)
,cte_numbered AS (
  SELECT ROW_NUMBER() OVER (PARTITION BY row_hash ORDER BY row_hash) AS occurrence, *
  FROM cte_hashed
)
SELECT
  ROW_NUMBER() OVER (ORDER BY school_urn, term, year) AS row_id
  ,* EXCEPT (row_hash, occurrence)
FROM cte_numbered
WHERE occurrence = 1;

-- Preview the deduped data
SELECT * FROM schools_no_exact_dupes LIMIT 10;

Each surviving row has also been assigned a `row_id` surrogate key — a system-generated identifier with no real-world meaning, useful for tracking individual rows through downstream steps.

Below we verify that the exact duplicate rows have been removed by counting the rows in the source data compared to the dataset after the deduplication step.

In [0]:
-- Verify: compare row counts before and after exact deduplication
SELECT 'Before (schools_reconciled)' AS stage, COUNT(*) AS rows
FROM silver.schools_reconciled

UNION ALL

SELECT 'After (exact dupes removed)', COUNT(*)
FROM schools_no_exact_dupes;

## Step 2 — Checking cardinality

With exact duplicates removed, we can now meaningfully test whether a column (or combination of columns) uniquely identifies each row.

**Cardinality** describes the grain of your dataset — what one row corresponds to in the real world. The simplest check: compare `COUNT(*)` to `COUNT(DISTINCT candidate_key)`. If they match, each row is unique at that grain.

A **candidate key** is any column (or set of columns) that *could* serve as the primary key — it uniquely identifies every row with no nulls or duplicates. A table can have several candidate keys; the one you choose to enforce becomes the primary key, and the rest remain candidates. Our job in this step is to test which columns qualify.

We'll check both **schools** and **pupils**, starting with the schools data we just deduped.

In [0]:
-- Cardinality check: is school_urn unique in the deduped data?
SELECT
  COUNT(*) AS total_rows
  ,COUNT(DISTINCT school_urn) AS distinct_school_urns
  ,COUNT(*) - COUNT(DISTINCT school_urn) AS remaining_duplicates
  ,CASE
    WHEN COUNT(*) = COUNT(DISTINCT school_urn)
    THEN 'school_urn is unique — one row per school'
    ELSE 'Duplicates on school_urn remain — these are natural key conflicts'
  END AS result
FROM schools_no_exact_dupes;

The result shows far more rows than distinct `school_urn` values — but this doesn't necessarily mean the data is dirty. Remember, `schools_reconciled` contains **four termly snapshots**. The same school appearing in multiple snapshots is entirely expected.

Let's test whether `school_urn` + `term` + `year` forms a unique composite key — i.e. whether the grain is **one row per school per term per year**.

In [0]:
-- Check composite key: is school_urn + term + year unique?
-- If so, the grain is one row per school per snapshot
SELECT
  COUNT(*) AS total_rows
  ,COUNT(DISTINCT CONCAT(school_urn, '|', term, '|', year)) AS distinct_school_snapshot
  ,COUNT(*) - COUNT(DISTINCT CONCAT(school_urn, '|', term, '|', year)) AS remaining_duplicates
  ,CASE
    WHEN COUNT(*) = COUNT(DISTINCT CONCAT(school_urn, '|', term, '|', year))
    THEN 'One row per school per term per year — grain confirmed'
    ELSE 'Duplicates remain — genuine within-snapshot conflicts'
  END AS result
FROM schools_no_exact_dupes;

Adding `term` and `year` to the key resolves almost all duplicates — confirming the grain is **one row per school per term per year**.

The small number of remaining duplicates are **genuine within-snapshot conflicts**: the same school appearing more than once in the same term extract, with differing values (e.g. a corrected name or updated status). These will need a business rule to resolve in **Notebook 06**.

Let’s investigate what actually differs between these duplicate rows.

In [0]:
-- Investigate: for schools that appear more than once per snapshot, what differs?
SELECT
  school_urn
  ,COUNT(*) AS occurrences
  ,COUNT(DISTINCT CONCAT(term, '|', year)) AS distinct_snapshots
  ,COUNT(DISTINCT school_name) AS distinct_names
  ,COUNT(DISTINCT status) AS distinct_statuses
FROM schools_no_exact_dupes
GROUP BY school_urn
HAVING COUNT(*) > COUNT(DISTINCT CONCAT(term, '|', year))
ORDER BY occurrences DESC
LIMIT 10;

### Schools grain confirmed

The composite key `school_urn` + `term` + `year` is almost unique — the grain is **one row per school per term per year**, with just a small number of genuine within-snapshot conflicts to resolve in **Notebook 06**.

This is a common pattern in real-world data: what initially looks like a sea of duplicates turns out to be mostly **expected repetition** (snapshots), with only a handful of genuine conflicts underneath.

### Checking cardinality on the pupils data

Now let's apply the same diagnostic steps to the **pupils** dataset. First we remove exact duplicates from `pupils_reconciled`, then check whether `pupil_id` — or `pupil_id` + `term` + `year` — uniquely identifies each row.

In [0]:
-- Remove exact duplicates from pupils (same CONCAT_WS + ROW_NUMBER technique)
-- Assign row_id after dedup
CREATE OR REPLACE TEMP VIEW pupils_no_exact_dupes AS
WITH cte_hashed AS (
  SELECT CONCAT_WS('|', *) AS row_hash, *
  FROM silver.pupils_reconciled
)
,cte_numbered AS (
  SELECT ROW_NUMBER() OVER (PARTITION BY row_hash ORDER BY row_hash) AS occurrence, *
  FROM cte_hashed
)
SELECT
  ROW_NUMBER() OVER (ORDER BY pupil_id, term, year) AS row_id
  ,* EXCEPT (row_hash, occurrence)
FROM cte_numbered
WHERE occurrence = 1;

SELECT * FROM pupils_no_exact_dupes LIMIT 10;

With exact duplicates removed from the pupils data, let's now check whether `pupil_id` uniquely identifies each row.

In [0]:
-- Cardinality check: is pupil_id unique in the deduped pupils data?
SELECT
  COUNT(*) AS total_rows
  ,COUNT(DISTINCT pupil_id) AS distinct_pupil_ids
  ,COUNT(*) - COUNT(DISTINCT pupil_id) AS remaining_duplicates
FROM pupils_no_exact_dupes;

More rows than distinct `pupil_id` values — but as with schools, this is largely expected: we have four termly snapshots, so each pupil appearing in every term accounts for roughly 4 rows.

Let's check whether `pupil_id` + `term` + `year` resolves it to one row per pupil per snapshot.

In [0]:
-- Check composite key: is pupil_id + term + year unique?
SELECT
  COUNT(*) AS total_rows
  ,COUNT(DISTINCT CONCAT(pupil_id, '|', term, '|', year)) AS distinct_pupil_snapshot
  ,COUNT(*) - COUNT(DISTINCT CONCAT(pupil_id, '|', term, '|', year)) AS remaining_duplicates
  ,CASE
    WHEN COUNT(*) = COUNT(DISTINCT CONCAT(pupil_id, '|', term, '|', year))
    THEN 'One row per pupil per snapshot'
    ELSE 'Duplicates remain even within individual snapshots'
  END AS result
FROM pupils_no_exact_dupes;

Unlike schools — where the composite key was almost clean — the pupils data has **17 remaining duplicates** after adding `term` and `year` to the key. That means 17 rows share a `pupil_id` + `term` + `year` combination with at least one other row in the same snapshot.

The grain is still **one row per pupil per term per year** — these aren't additional snapshots or expected repetition. They're genuine within-snapshot conflicts: the same pupil appearing twice in the same extract with different values.

Let's find out which pupils are affected and in which snapshots.

In [0]:
-- Which pupils have within-snapshot conflicts, and in which term/year?
-- A pupil with 2 rows in the same snapshot has a genuine conflict to resolve
SELECT
  pupil_id
  ,term
  ,year
  ,COUNT(*) AS rows_in_snapshot
FROM pupils_no_exact_dupes
GROUP BY pupil_id, term, year
HAVING COUNT(*) > 1
ORDER BY year, term, pupil_id;

The 17 within-snapshot conflicts are spread across **all four snapshots**, affecting 15 distinct pupils (P002 has conflicts in both autumn 2024 *and* summer 2025). Every conflict involves exactly 2 rows — the same pupil appearing twice in the same extract.

But what actually **differs** between the two rows? Let's pull a few examples side by side to see the kinds of conflicts we're dealing with.

In [0]:
-- Show the actual conflicting rows for a sample of affected pupils
-- Pick one example from each snapshot to illustrate the variety of conflicts
SELECT
  pupil_id
  ,first_name
  ,last_name
  ,term
  ,year
  ,school_urn
  ,metadata_json:sen_status AS sen_status
  ,metadata_json:fsm_eligible AS fsm_eligible
  ,metadata_json:contact.parent_name AS parent_name
FROM pupils_no_exact_dupes
WHERE (pupil_id, term, year) IN (
  SELECT pupil_id, term, year
  FROM pupils_no_exact_dupes
  GROUP BY pupil_id, term, year
  HAVING COUNT(*) > 1
)
ORDER BY pupil_id, term, year;

Scanning the pairs side by side, several distinct **types of conflict** emerge:

| Conflict type | Examples | What differs |
| --- | --- | --- |
| **Name variant** | P002 (autumn 2024) | `Bob` vs `Robert` — nickname vs legal name |
| **Casing inconsistency** | P011 (autumn 2024) | `katie JONES` vs `Katie Jones` — same person, different entry |
| **FSM eligibility flipped** | P005, P006, P007, P009 | `fsm_eligible` is `true` in one row, `false` in the other |
| **SEN status changed** | P008, P010, P015, P020, P022 | `sen_status` differs — e.g. `EHCP` vs `SEN Support`, or `SEN Support` vs `None` |
| **Different parent contact** | P011, P016 | Different `parent_name` — e.g. mother vs father listed as primary |
| **Multiple fields conflict** | P020 (summer 2025) | Both `sen_status` *and* `fsm_eligible` differ between the two rows |
| **Hidden in JSON only** | P002 (summer), P003, P013, P014, P017 | Visible columns look identical — the conflict is in `metadata_json` fields like `attendance_pct`, `allergy_info`, or `secondary_contact` |

Every one of these is a **genuine within-snapshot conflict** — the same pupil appearing twice in the same termly extract with contradictory information. Unlike exact duplicates, these cannot be resolved mechanically. Each requires a **business rule** to decide which row to keep (or how to merge them).

### What have we learned about the grain?

The cardinality checks confirm that both datasets follow the same pattern: the grain is **one row per entity per term per year**, with a number of genuine within-snapshot conflicts.

* **Schools**: a small number of conflicts (same pattern, different scale)
* **Pupils**: 17 conflicts across 15 distinct pupils and all four snapshots

These natural key duplicates will be resolved with business rules in **Notebook 06 — Removing Duplicates**.

## Step 3 — Understanding keys

Keys are the columns (or combinations of columns) that **identify and connect** rows. We've already been working with them throughout this notebook — let's now put names to the concepts.

| Key type | Definition | In our data |
| --- | --- | --- |
| **Primary key** | Column(s) that **uniquely identify** every row. No nulls, no duplicates. | `school_urn` + `term` + `year` for schools; `pupil_id` + `term` + `year` for pupils. In the **original source systems**, `school_urn` and `pupil_id` alone would be primary keys — each snapshot has one row per school or pupil. The composite key is only necessary here because we **concatenated multiple snapshots** into a single table. |
| **Natural key** | Derived from **real-world attributes**, meaningful to humans. | `school_urn` and `pupil_id` identify the real-world entity. `term` and `year` identify the point in time. All four are natural keys — they carry real-world meaning. In **Notebook 04** we extracted `term` and `year` from `source_table` specifically to make these dimensions explicit rather than encoded in a single string. |
| **Surrogate key** | **System-generated** with no real-world meaning. Stable even if real-world attributes change. | The `row_id` column assigned after exact deduplication (Step 1 above) is a surrogate key — generated by `ROW_NUMBER()`, it gives every surviving row a stable, meaningless identifier. It's assigned *after* dedup so that only clean rows receive an ID — assigning it beforehand would make every row appear unique and break the duplicate detection. |
| **Composite key** | Two or more columns that together uniquely identify a row, even when neither is unique alone. | Both primary keys above are composite — neither `school_urn` nor `term` nor `year` is unique on its own, but together they are. |
| **Foreign key** | A column (or columns) in one table that references the **primary key** of another, creating a relationship between them. | `school_urn` + `term` + `year` in `pupils_reconciled` references `school_urn` + `term` + `year` in `schools_reconciled`. Joining on `school_urn` alone would match a pupil to **every snapshot** of their school, fanning out rows and inflating counts. |

> **These categories are not mutually exclusive.** A key can be described by more than one of the terms above at the same time. In our data, the primary key is *also* a composite key (it takes three columns to achieve uniqueness) and is built from *natural* keys (real-world identifiers rather than system-generated values). The surrogate key `row_id` is the only key that falls into just one category. Recognising that these labels describe different *properties* of a key — rather than different keys — helps avoid confusion.

> **Operational vs analytical systems:** In operational systems (databases powering applications), primary keys are almost always a **single column** — typically an auto-incrementing ID or a natural identifier like `school_urn`. In **analytical datasets** like ours, composite primary keys are far more common because we routinely combine data from multiple time periods, sources, or systems into a single table. Understanding this distinction helps explain why your cardinality checks keep finding "duplicates" — they're not errors, they're a natural consequence of the analytical data model.

## Step 4 — Validating referential integrity

So far we've focused on understanding each table in isolation. But schools and pupils are connected — `school_urn` in the pupils table references `school_urn` in the schools table. This is a **foreign key** relationship, and it can break.

When a foreign key value in one table doesn't exist in the referenced table, you have an **orphaned foreign key**. These silently drop rows from inner joins — a pupil referencing a school that doesn't appear in the schools table simply vanishes from your results with no error or warning.

Since the primary key is composite (`school_urn` + `term` + `year`), the FK check must match on **all three** columns. Joining on `school_urn` alone would match a pupil to every snapshot of their school, which defeats the purpose.

In [0]:
-- Find orphaned foreign keys: pupils whose school doesn't exist in the same snapshot
-- Join on all three components of the composite key
SELECT
  p.pupil_id
  ,p.first_name
  ,p.last_name
  ,p.school_urn AS orphaned_school_urn
  ,p.term
  ,p.year
FROM pupils_no_exact_dupes p
LEFT JOIN schools_no_exact_dupes s
  ON p.school_urn = s.school_urn
  AND p.term = s.term
  AND p.year = s.year
WHERE s.school_urn IS NULL;

### Handling orphaned records — a business decision

Once you've identified orphaned foreign keys, you need to decide what to do with them. Those records can either be removed from the dataset , or if you want to retain the data would be to add a record in the schools data that has a `school_urn` of `999999`.

Removing or adding data is is a **business decision** that should be:

* **Documented clearly** — record *what* was added or removed, *why*, and *how many rows* were affected
* **Proportionate** — if orphaned rows represent a significant share of the data, additions or removals  could introduce bias
* **Reversible** — filter rows out rather than deleting them, so the decision can be revisited
* **Communicated** — stakeholders should know that certain records were added or excluded and understand the impact

> **Good practice:** Add a comment or markdown cell explaining the rationale whenever you modify your data.

## Summary — What did we learn?

This notebook established four essential diagnostic steps:

1. **Exact duplicates removed** — using `CONCAT_WS` + `ROW_NUMBER`, we mechanically eliminated carbon-copy rows with no domain knowledge required
2. **Cardinality checked** — the grain is one row per entity per `term` per `year`, with a small number of genuine within-snapshot conflicts to resolve
3. **Keys understood** — primary (`school_urn` + `term` + `year`), natural (`school_urn`, `term`, `year`), surrogate (`row_id`), composite, and foreign keys each play a different role (albeit in this example the primary and natural keys are the same)
4. **Referential integrity validated** — orphaned foreign keys identified using `LEFT JOIN` + `IS NULL` on the full composite key

The contents of the `schools_no_exact_dupes` and `pupils_no_exact_dupes` have been saved to the following tables in the silver layer to be picked up in the next notebook:

* `catalog_40_copper_analyst_training.silver.schools_entire_duplicates_removed`
* `catalog_40_copper_analyst_training.silver.pupils_entire_duplicates_removed`

### What's next

* **Notebook 06 — Removing Duplicates** will resolve the natural key conflicts using business rules and `ROW_NUMBER` with meaningful ordering
* **Notebook 07 — Standardising Fields and Labels** will clean the actual values (casing, whitespace, abbreviations)
* **Notebook 08 — Building the Gold Layer** will apply all steps end-to-end